# Perch Watch — Model Training

Two-stage pipeline:
1. **Stage 1 — generic detector.** Single class (`bird`), trained from stock COCO-pretrained `yolo11n.pt`. This is what actually drives the turret's trigger.
2. **Stage 2 — species classifier.** 11 classes, trained via transfer learning from the Stage 1 weights (not from scratch). Runs on crops from Stage 1's detections, for logging/identification only. See the main README for why the two are kept separate rather than merged into one model.

Both stages use the same Roboflow workspace, different projects/versions. You will be able to find these datasets within Roboflow under my name `Jonathan Ferman-Ramirez` with titles (`birds`) and (`Umatilla_County_Birds`) 

## Setup

In [ ]:
!pip install -q ultralytics roboflow


## Stage 1 — Generic Bird Detector

Trained from scratch (stock COCO-pretrained weights), single `bird` class. Dataset includes the original flying-pose images, added natural/perched-pose images, and hard-negative background images (flowers, wooden benches/logs, stoplights) added after evaluation caught false positives on those.

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="YOUR_API_KEY")
project = rf.workspace("jonathans-workspace-tdeuv").project("birds-imhhr")
version = project.version(2)
dataset = version.download("yolov11")


In [ ]:
from ultralytics import YOLO

# Fresh COCO-pretrained
model = YOLO("yolo11n.pt")

results = model.train(data=f"{dataset.location}/data.yaml", epochs=50, imgsz=640, patience=10)


### Stage 1 evaluation

In [ ]:
metrics = model.val(data=f"{dataset.location}/data.yaml")

print(f"mAP50: {metrics.box.map50}")
print(f"mAP50-95: {metrics.box.map}")
print(f"Precision: {metrics.box.mp}")
print(f"Recall: {metrics.box.mr}")

# Reference — final numbers from the last full run:
# Precision 0.877, Recall 0.841, mAP50 0.883, mAP50-95 0.492


### Export Stage 1 weights

In [ ]:
from google.colab import files
files.download("runs/detect/train/weights/best.pt")


## Stage 2 — Species Classifier

Transfer learning **from the Stage 1 weights above**, not from COCO weights. This is what carries over the general "what does a bird look like" features as a head start.

Covers 11 species. Class head is reinitialized automatically for the new class count (`nc=11`). watch for the `Transferred X/Y items from pretrained weights` message confirming the backbone carried over correctly.

In [ ]:
project = rf.workspace("jonathans-workspace-tdeuv").project("umatilla_county_birds")
version = project.version(2)
dataset = version.download("yolov11")


In [ ]:
from ultralytics import YOLO

model = YOLO("/content/best.pt")  # Stage 1 weights

results = model.train(data=f"{dataset.location}/data.yaml", epochs=40, imgsz=640, patience=10)


### Stage 2 evaluation

In [ ]:
metrics = model.val(data=f"{dataset.location}/data.yaml")

print(f"mAP50: {metrics.box.map50}")
print(f"mAP50-95: {metrics.box.map}")
print(f"Precision: {metrics.box.mp}")
print(f"Recall: {metrics.box.mr}")

# Reference — final numbers after fixing the Bufflehead/bald-eagle mislabeling:
# Precision 0.49 (approx), Recall 0.498, mAP50 0.405, mAP50-95 0.283
# Check the confusion matrix too (auto-saved to runs/detect/val/confusion_matrix.png) —
# the four goose species (Anser-Albifrons, Anser-Caerulescens, Branta-Canadensis,
# Branta-Hutchinsii) are a known hard-to-separate cluster, not a labeling bug.


### Export Stage 2 weights

In [ ]:
files.download("runs/detect/train2/weights/best.pt")


## Export both models to NCNN for Pi deployment

NCNN was benchmarked faster than OpenVINO on the Pi 5 for both models (generic: 11.60 vs 7.10 FPS; species: 11.43 vs 7.83 FPS, live webcam)

In [ ]:
stage1_model = YOLO("stage1_generic_best.pt")  # path to whichever weights file you're exporting
stage1_model.export(format="ncnn", task="detect")

stage2_model = YOLO("stage2_species_best.pt")
stage2_model.export(format="ncnn", task="detect")
